In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

# --- Approach 1: Quick built-in (for tensors already in memory) ---
X = torch.randn(100, 5)          # 100 samples, 5 features
y = torch.randint(0, 3, (100,))  # 3-class labels
quick_dataset = TensorDataset(X, y)
print(f"TensorDataset size: {len(quick_dataset)}")
print(f"Sample 0: features={quick_dataset[0][0].shape}  label={quick_dataset[0][1].item()}")

TensorDataset size: 100
Sample 0: features=torch.Size([5])  label=0


In [2]:
# --- Approach 2: Custom Dataset (for files, images, text, etc.) ---
class ClassificationDataset(Dataset):
    """Custom dataset for tabular classification data."""
    def __init__(self, n_samples=200, n_features=5, n_classes=3):
        torch.manual_seed(42)
        self.features = torch.randn(n_samples, n_features)
        # Labels: which feature dimension has the max value
        self.labels = self.features.argmax(dim=1) % n_classes

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

dataset = ClassificationDataset(n_samples=200, n_features=5)
print(f"\nCustom dataset: {len(dataset)} samples")
print(f"Feature shape: {dataset[0][0].shape}  Label: {dataset[0][1].item()}")

# --- DataLoader: batching, shuffling, multiprocessing ---
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,         # shuffle each epoch
    drop_last=False,      # keep partial last batch
)

# Iterate over batches
for batch_idx, (batch_X, batch_y) in enumerate(loader):
    if batch_idx == 0:
        print(f"\nBatch 0: X={batch_X.shape}  y={batch_y.shape}")  # [32, 5] [32]
    if batch_idx == len(loader) - 1:
        print(f"Last batch ({batch_idx}): X={batch_X.shape}  (may be < 32)")

print(f"Total batches: {len(loader)}")  # ceil(200/32) = 7

# --- DataLoader as an iterator (useful for large datasets) ---
loader_iter = iter(DataLoader(dataset, batch_size=50))
first_batch = next(loader_iter)
print(f"\nIterator batch: {first_batch[0].shape}")  # [50, 5]


Custom dataset: 200 samples
Feature shape: torch.Size([5])  Label: 0

Batch 0: X=torch.Size([32, 5])  y=torch.Size([32])
Last batch (6): X=torch.Size([8, 5])  (may be < 32)
Total batches: 7

Iterator batch: torch.Size([50, 5])
